### Imports

In [1]:
import itertools
import pandas as pd
import numpy as np
from scipy.optimize import minimize
from scipy.stats import norm
import matplotlib 
import matplotlib.pyplot as plt
%matplotlib inline

### Reprise du code de 2_1_bradley_terry 

In [2]:
df_votes = pd.read_parquet("../votes.parquet")
df_filtered = df_votes[df_votes["both_equal"] == False]

In [10]:
def get_occurences_model(df:pd.DataFrame) -> pd.Series:
    """Récupère les modèles et leur nombre d'occurrences dans les colonnes `model_a_name` et `model_b_name`."""
    model_names_a = df["model_a_name"].value_counts()
    model_names_b = df["model_b_name"].value_counts()

    # Mise en commun des deux séries
    return model_names_a.add(model_names_b, fill_value=0).sort_values(ascending=False)

def build_ranking(model_names:list[str], df:pd.DataFrame) -> np.ndarray:
    """
    Construit la matrice de gain où W[i, j] est le nombre de fois que le modèle i a été préféré au modèle j dans les votes.
    """
    gains = np.zeros((len(model_names), len(model_names)), dtype=int)
    for row in df.itertuples():
        if row.model_a_name in model_names and row.model_b_name in model_names:
            i = model_names.index(row.model_a_name)
            j = model_names.index(row.model_b_name)
            if i == j:
                continue
            if row.chosen_model_name == row.model_a_name:
                gains[i, j] += 1
            elif row.chosen_model_name == row.model_b_name:
                gains[j, i] += 1
    return gains

def estimate_bradley_terry(W:np.ndarray):
    """
    Estime les paramètres β du modèle de Bradley-Terry à partir de la matrice de gains W.

    Args:
        W (np.ndarray): Matrice de gains, W[i,j] = nombre de fois que i bat j.

    Returns:
        np.ndarray: Paramètres β estimés pour chaque modèle.
        np.ndarray: Classement des modèles (indices des modèles par ordre décroissant de β).
    """
    n_models = W.shape[0]
    beta_init = np.ones(n_models)

    def log_likelihood(beta, W):
        likelihood = 0.0
        for i in range(n_models):
            for j in range(n_models):
                if i != j:
                    p = beta[i] / (beta[i] + beta[j])
                    likelihood += W[i, j] * np.log(p)
        return -likelihood  # On minimise l'opposé de la log-vraisemblance

    result = minimize(log_likelihood, beta_init, args=(W,), method='L-BFGS-B')
    beta_estimated = result.x
    ranking = np.argsort(beta_estimated)[::-1]

    return beta_estimated, ranking

## Exercice 2.2

### Transitivité stochastique

On cherche à vérifier la transitivité stochastique du modèle de Bradley-Terry sur le sous-graphe des 20 modèles les plus comparés. La définition de transitivité stochastique retenue ici pour ce test est celle de la Weak Stochastic Transitivity :

Pour $s \in [0,1], P(i>j) > s \land P(j>k) > s \implies P(i,k) > s$

In [4]:
def test_stochastic_transitivity(beta: np.ndarray, threshold: float = 0.5) -> dict:
    """
    Teste la transitivité stochastique (Weak Stochastic Transitivity) sur le threshold à partir des paramètres beta du modèle de Bradley-Terry.

    Args:
        beta (np.ndarray): paramètres estimés de Bradley-Terry
        threshold (float): seuil de décision (0.5 par défaut)

    Returns:
        dict: métriques de transitivité
    """
    n = len(beta)

    total_triplets = 0
    transitive = 0
    violations = 0

    # Probabilité de Bradley-Terry
    def p(i, j):
        return beta[i] / (beta[i] + beta[j])

    for i, j, k in itertools.permutations(range(n), 3):

        pij = p(i, j)
        pjk = p(j, k)
        pik = p(i, k)

        # Condition de transitivité stochastique
        if pij > threshold and pjk > threshold:
            total_triplets += 1

            if pik > threshold:
                transitive += 1
            else:
                violations += 1

    return {
        "total_triplets": total_triplets,
        "transitive": transitive,
        "violations": violations,
        "transitivity_rate": transitive / total_triplets if total_triplets > 0 else np.nan
    }

In [16]:
# Top 20 most compared models
model_names = get_occurences_model(df_filtered)
top_20_models = model_names.head(20).index.tolist()

# Bradley-Terry parameters
W = build_ranking(top_20_models, df_filtered)
beta, ranking = estimate_bradley_terry(W)

# Test de transitivité
for i in range(10):
    results = test_stochastic_transitivity(beta, i*0.1)
    print(f"Seuil : {i/10}, results : {results}")


Seuil : 0.0, results : {'total_triplets': 6840, 'transitive': 6840, 'violations': 0, 'transitivity_rate': 1.0}
Seuil : 0.1, results : {'total_triplets': 6840, 'transitive': 6840, 'violations': 0, 'transitivity_rate': 1.0}
Seuil : 0.2, results : {'total_triplets': 6552, 'transitive': 6438, 'violations': 114, 'transitivity_rate': 0.9826007326007326}
Seuil : 0.3, results : {'total_triplets': 5076, 'transitive': 4750, 'violations': 326, 'transitivity_rate': 0.9357762017336485}
Seuil : 0.4, results : {'total_triplets': 2886, 'transitive': 2742, 'violations': 144, 'transitivity_rate': 0.9501039501039501}
Seuil : 0.5, results : {'total_triplets': 1140, 'transitive': 1140, 'violations': 0, 'transitivity_rate': 1.0}
Seuil : 0.6, results : {'total_triplets': 186, 'transitive': 186, 'violations': 0, 'transitivity_rate': 1.0}
Seuil : 0.7, results : {'total_triplets': 0, 'transitive': 0, 'violations': 0, 'transitivity_rate': nan}
Seuil : 0.8, results : {'total_triplets': 0, 'transitive': 0, 'violat

Globalement, pour tous les seuils testés, les taux de transitivité sont très élevé, avec un minimum à 94% : le modèle de Bradley-Terry tend à respecter la transitivité stochastique.

### Test de puissance pour distinction statistique

In [17]:
def required_comparisons(p, alpha=0.05, power=0.8):
    """
    Calcule le nombre de comparaisons nécessaires pour détecter p > 0.5 avec un test binomial
    """
    z_alpha = norm.ppf(1 - alpha / 2)
    z_power = norm.ppf(power)

    numerator = (z_alpha + z_power) ** 2 * p * (1 - p)
    denominator = (p - 0.5) ** 2

    n = numerator / denominator
    return int(np.ceil(n))

In [18]:
# Probabilité de Bradley-Terry
p = beta[2] / (beta[2] + beta[4])

# Nombre de comparaisons minimum requises
n_required = required_comparisons(p)

print(f"p = {p}")
print(f"Nombre minimum de comparaisons : {n_required}")

p = 0.3128668726098559
Nombre minimum de comparaisons : 49


Il faut un très grand nombre de comparaisons pour pouvoir distinguer statistiquement le modèle 3 du modèle 5 !

### Extension de Davidson

Pour pouvoir prendre en compte les votes ex-aequo entre deux modèles dans le modèle de Bradley-Terry, on peut implémenter l'extension de Davidson.

In [ ]:
def build_ranking_davidson(model_names, df):
    """
    Construit les matrices W et T, où W[i, j] est le nombre de fois que le modèle i a été préféré au modèle j dans les votes,
    et T[i, j] est le nombre de fois que le modèle i a été égal au modèle j dans les votes.
    """
    n = len(model_names)

    W = np.zeros((n, n), dtype=int)
    T = np.zeros((n, n), dtype=int)

    for row in df.itertuples():
        if row.model_a_name in model_names and row.model_b_name in model_names:
            i = model_names.index(row.model_a_name)
            j = model_names.index(row.model_b_name)

            if i == j:
                continue

            if row.both_equal:
                T[i, j] += 1
                T[j, i] += 1
            else:
                if row.chosen_model_name == row.model_a_name:
                    W[i, j] += 1
                elif row.chosen_model_name == row.model_b_name:
                    W[j, i] += 1

    return W, T

def estimate_davidson(W, T):
    """
    Estime les paramètres beta du modèle de Bradley-Terry à partir des matrices W et T.

    Args:
        W (np.ndarray): Matrice de gains, W[i,j] = nombre de fois que i bat j.
        T (np.ndarray): Matrice de gains, T[i,j] = nombre de fois que i est j sont égaux.

    Returns:
        np.ndarray: Paramètres beta estimés pour chaque modèle.
        float: Paramètre nu.
        np.ndarray: Classement des modèles (indices des modèles par ordre décroissant de beta).
    """

    n = W.shape[0]

    # Initialisation de beta et nu 
    x0 = np.ones(n + 1)
    x0[-1] = 0.5

    def neg_log_likelihood(x):
        beta = x[:n]
        nu = x[-1]

        if np.any(beta <= 0) or nu <= 0:
            return np.inf

        likelihood = 0.0

        for i in range(n):
            for j in range(i + 1, n):

                wij = W[i, j]
                wji = W[j, i]
                tij = T[i, j]

                if wij + wji + tij == 0:
                    continue

                denom = beta[i] + beta[j] + nu * np.sqrt(beta[i] * beta[j])

                if wij > 0:
                    likelihood += wij * np.log(beta[i] / denom)
                if wji > 0:
                    likelihood += wji * np.log(beta[j] / denom)
                if tij > 0:
                    likelihood += tij * np.log(nu * np.sqrt(beta[i] * beta[j]) / denom)

        return -likelihood # On minimise l'opposé de la log-vraisemblance

    result = minimize(neg_log_likelihood, x0, method='L-BFGS-B')

    beta = result.x[:n]
    nu = result.x[-1]

    ranking = np.argsort(beta)[::-1]

    return beta, nu, ranking

In [14]:
model_names = get_occurences_model(df_votes)
top_20_models = model_names.head(20).index.tolist()

# Bradley-Terry parameters
W_bradley = build_ranking(top_20_models, df_votes)
beta_bradley, ranking_bradley = estimate_bradley_terry(W_bradley)

# Davidson parameters
W_davidson, T_davidson = build_ranking_davidson(top_20_models, df_votes)
beta_davidson, nu_davidson, ranking_davidson = estimate_davidson(W_davidson, T_davidson)

for i in range(len(beta_bradley)):
    print(f"Beta Bradley-Terry : {beta_bradley[i]}; Beta Davidson {beta_davidson[i]}; différence : {beta_bradley[i]-beta_davidson[i]}")
print(f"Nu Davidson : {nu_davidson}")

C:\Users\MickaelMartinelliCS\AppData\Local\Temp\ipykernel_27932\2585238814.py:46: RuntimeWarning: invalid value encountered in log
  likelihood += W[i, j] * np.log(p)


Beta Bradley-Terry : 0.7929846907920807; Beta Davidson 0.8107920076634317; différence : -0.017807316871351064
Beta Bradley-Terry : 0.6852801851003096; Beta Davidson 0.7143780134343046; différence : -0.02909782833399499
Beta Bradley-Terry : 0.9076821672748873; Beta Davidson 0.9223605360851479; différence : -0.014678368810260634
Beta Bradley-Terry : 1.1207587183448597; Beta Davidson 1.0974359368442677; différence : 0.023322781500592082
Beta Bradley-Terry : 0.8736461586892903; Beta Davidson 0.886782605991154; différence : -0.013136447301863696
Beta Bradley-Terry : 0.8298184304817439; Beta Davidson 0.8462595173213427; différence : -0.016441086839598884
Beta Bradley-Terry : 1.3081919452780102; Beta Davidson 1.2408186617048544; différence : 0.06737328357315575
Beta Bradley-Terry : 1.1837030884155368; Beta Davidson 1.149557044083446; différence : 0.0341460443320909
Beta Bradley-Terry : 1.1184277846215616; Beta Davidson 1.0890117890154871; différence : 0.02941599560607444
Beta Bradley-Terry : 

C:\Users\MickaelMartinelliCS\AppData\Roaming\Python\Python313\site-packages\scipy\optimize\_numdiff.py:596: RuntimeWarning: invalid value encountered in subtract
  df = fun(x1) - f0


On observe globalement des scores $\beta$ très proches, que l'on utilise l'extension de Davidson ou non. On peut ainsi conclure que le modèle de Bradley-Terry reste plutôt robuste aux ex-aequo. Cependant, on remarque aussi un autre phénomène : pour les plus grandes valeurs de $\beta$, la différence se creuse entre le modèle standard et l'extension de Davidson. La prise en compte des égalités nuance donc plus fortement ces modèles à haute estimation de force créative. Ainsi, il semblerait que la prise en compte des égalités n'est pas très importante pour les modèles "clivants", où les votes s'opposants sont suffisants pour rendre compte du phénomène, mais l'est davantage pour les modèles qui font déjà conscensus, car elle permet d'affiner en nuançant les résultats.